In [6]:
%%writefile research_server.py

import arxiv
import json
import os
from typing import List
from mcp.server.fastmcp import FastMCP

PAPER_DIR = "papers"

# initialize FastMCP server
mcp = FastMCP("research")

@mcp.tool()
def search_papers(topic: str, max_results: int = 5) -> List[str]:
    """Search for papers on arXiv based on a topic. and store their information
    
    Args:
        topic (str): The topic to search for.
        max_results (int): The maximum number of results to retrive (default: 5).
        
    Returns:
        List[str]: A list of paper IDs found in the search
    """
    
    # use arxiv library to search for papers
    client = arxiv.Client(page_size=max_results, delay_seconds=3.0, num_retries=3)
    
    # Search for the most relevant articles matching the querid topic and get the top results
    search = arxiv.Search(
        query = topic, 
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance
        )
    
    papers = client.results(search)
    
    # Create the papers directory if it doesn't exist
    path = os.path.join(PAPER_DIR, topic.lower().replace(" ", "_"))
    
    os.makedirs(path, exist_ok=True)
    
    file_path = os.path.join(path, "papers_info.json")
    
    # Try to load existing papers info
    try:
        with open(file_path, "r") as json_file:
            papers_info = json.load(json_file)
    except (FileNotFoundError, json.JSONDecodeError):
        papers_info = {}
        
    paper_ids = []
    for paper in papers:
        paper_ids.append(paper.get_short_id()) 
        paper_data = {
            'title': paper.title,
            'authors': [author.name for author in paper.authors],
            'summary': paper.summary,
            'pdf_url': paper.pdf_url,
            'published': str(paper.published.date()),
        }
        papers_info[paper.get_short_id()] = paper_data
        
    # Save the updated papers info back to the JSON file
    
    with open(file_path, "w") as json_file:
        json.dump(papers_info, json_file, indent=2)
        
    print(f"Results are saved in: {file_path}")

    return paper_ids

@mcp.tool()
def extract_info(paper_id: str) -> str:
    """
    Search for information about a specific paper acros all topic directories.
    
    args:
        paper_id (str): The ID of the paper to search for.
    returns:
        JSPM string with paper information if found, otherwise a message indicating the paper was not found.
    """
    
    for item in os.listdir(PAPER_DIR):
        item_path = os.path.join(PAPER_DIR, item)
        if os.path.isdir(item_path):
            file_path = os.path.join(item_path, "papers_info.json")
            if os.path.exists(file_path):
                try:
                 with open(file_path, "r") as json_file:
                    papers_info = json.load(json_file)
                    if paper_id in papers_info:
                        return json.dumps(papers_info[paper_id], indent=2)
                except (json.JSONDecodeError, FileNotFoundError) as e:
                    print(f"Error reading {file_path}: {e}")
                    continue
    return f"There is no saved information related to the paper with ID: {paper_id}"

if __name__ == "__main__":
    # Initalize and run the server
    mcp.run(transport='stdio')

Overwriting research_server.py


In [10]:
# start a new terminal
import os
from IPython.display import IFrame

IFrame(f"{os.environ.get('DLAI_LOCAL_URL').format(port=8888)}terminals/1", 
       width=1024, height=768)

AttributeError: 'NoneType' object has no attribute 'format'